# Historical AskLaw prototype
Preserved for reference only; saved outputs and credentials have been removed.
Use the AtlasLex package and CLI. This notebook retains known prototype limitations and is not the application entry point.

In [ ]:
import os, json, math, re

from pathlib import Path
from typing import List, Dict, Any, Optional

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from google import genai


In [ ]:
DATA_DIR = Path("../examples/legacy_data")
ARTICLES_PATH = DATA_DIR / "articles.json"
ARTICLES_LOOKUP_PATH = DATA_DIR / "articles_lookup.json"
RAG_CHUNKS_PATH = DATA_DIR / "rag_chunks.json"
QDRANT_POINTS_PATH = DATA_DIR / "qdrant_points.json"
BM25_CORPUS_PATH = DATA_DIR / "bm25_corpus.json"


QDRANT_HOST = "localhost"
QDRANT_PORT = 6333
COLLECTION_NAME = "code_commerce"

#EMBED_MODEL = "intfloat/multilingual-e5-large"
EMBED_MODEL = "antoinelouis/french-me5-base"

DENSE_TOP_K = 10
SPARSE_TOP_K = 10
FINAL_TOP_K = 3

USE_QUERY_REWRITE = True
USE_ARTICLE_DEDUP = True


In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL, device="cpu")

def embed_query(texts: List[str]) -> List[List[float]]:
    prefixed = [f"query: {t}" for t in texts]
    return embed_model.encode(prefixed, normalize_embeddings=True).tolist()

def embed_passages(texts: List[str]) -> List[List[float]]:
    prefixed = [f"passage: {t}" for t in texts]
    return embed_model.encode(prefixed, normalize_embeddings=True).tolist()

In [ ]:
reranker = CrossEncoder("antoinelouis/crossencoder-me5-base-mmarcoFR", device="cpu")

def cross_encoder_rerank(query: str, chunks: List[Dict[str, Any]], top_k: Optional[int] = None) -> List[Dict[str, Any]]:
    pairs = []
    for c in chunks:
        text = f"{c.get('context_header', '')}\nArticle {c.get('article_number', '')}\n{c.get('text', '')}"
        pairs.append((query, text))

    scores = reranker.predict(pairs)

    reranked = []
    for c, s in zip(chunks, scores):
        c2 = dict(c)
        c2["_rerank_score"] = float(s)
        reranked.append(c2)

    reranked.sort(key=lambda x: x["_rerank_score"], reverse=True)

    if top_k is not None:
        reranked = reranked[:top_k]

    return reranked

In [ ]:
with open(ARTICLES_PATH, "r", encoding="utf-8") as f:
    articles = json.load(f)
    
with open(ARTICLES_LOOKUP_PATH, "r", encoding="utf-8") as f:
    articles_lookup = json.load(f)
    
with open(RAG_CHUNKS_PATH, "r", encoding="utf-8") as f:
    rag_chunks = json.load(f)

with open(QDRANT_POINTS_PATH, "r", encoding="utf-8") as f:
    qdrant_points = json.load(f)
    
with open(BM25_CORPUS_PATH, "r", encoding="utf-8") as f:
    bm25_corpus = json.load(f)
    
    
print(f"loaded {len(articles)} articles")
print(f"loaded {len(articles_lookup)} articles_lookup")
print(f"loaded {len(rag_chunks)} RAG chunks")
print(f"loaded {len(qdrant_points)} qdrant points")
print(f"loaded {len(bm25_corpus)} BM25 docs")


In [ ]:
# build lookup maps
chunk_lookup: Dict[str, Dict[str, Any]] = {
    str(doc["chunk_id"]): doc for doc in rag_chunks
}

def normalizeChunkID(x: Any) -> str:
    return str(x)

In [ ]:
# build bm25 index

def simpleTokenize(text: str) -> List[str]:
    text = text.lower()
    text = re.sub(r"[^\w\sàâäéèêëîïôöùûüç'-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()
bm25_ids = []
bm25_tokenized = []

for doc in bm25_corpus:
    bm25_ids.append(str(doc["chunk_id"]))
    bm25_tokenized.append(simpleTokenize(doc["text"]))

bm25 = BM25Okapi(bm25_tokenized)
print(f"BM25 index built over {len(bm25_ids)} chunks")

In [ ]:
# BM25 RETRIEVAL

def bm25Search(query: str, top_k: int=10) -> List[Dict[str, Any]]:
    tokens = simpleTokenize(query)
    scores = bm25.get_scores(tokens)
    
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]
    
    return [
        {
            "chunk_id": bm25_ids[i],
            "score": float(scores[i]),
            "source": "bm25"
        }
        for i in top_indices
    ]

In [ ]:
# connect to qdrant
qdrant_client = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
print(qdrant_client.get_collections())

In [ ]:
def create_collection_if_missing(vector_size: int):
    existing = [c.name for c in qdrant_client.get_collections().collections]

    if COLLECTION_NAME not in existing:
        qdrant_client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
        )
        print("Collection created")
    else:
        info = qdrant_client.get_collection(COLLECTION_NAME)
        existing_vectors = info.config.params.vectors

        if existing_vectors.size != vector_size:
            raise ValueError(
                f"Collection '{COLLECTION_NAME}' exists with vector size {existing_vectors.size}, "
                f"but current embeddings have size {vector_size}."
            )

        print("Collection already exists")


texts = [p["payload"]["prefixed_text"] for p in qdrant_points]
vectors = embed_passages(texts)

if not vectors:
    raise ValueError("No vectors were generated.")

create_collection_if_missing(len(vectors[0]))

points = []
for point, vec in zip(qdrant_points, vectors):
    points.append(
        PointStruct(
            id=normalizeChunkID(point["id"]),
            vector=vec,
            payload=point["payload"],
        )
    )

batch_size = 128
for i in range(0, len(points), batch_size):
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i + batch_size],
    )

print(f"Inserted {len(points)} points into Qdrant")

In [ ]:
# dense retrival

def denseSearch(query: str, top_k: int = 10) -> List[Dict[str, Any]]:
    query_vector = embed_query([query])[0]

    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    )

    results = []
    for h in response.points:
        cid = normalizeChunkID(h.id)
        payload = h.payload if h.payload is not None else chunk_lookup.get(cid, {})
        results.append({
            "chunk_id": cid,
            "score": float(h.score),
            "payload": payload,
            "source": "dense",
        })

    return results

In [ ]:
# Reciprocal Rank Fusion, combines the dense and sparse retrivals
def reciprocal_rank_fusion(
    dense_results: List[Dict[str, Any]],
    sparse_results: List[Dict[str, Any]],
    k: int = 60,
) -> List[str]:
    scores: Dict[str, float] = {}

    for rank, result in enumerate(dense_results):
        cid = normalizeChunkID(result["chunk_id"])
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)

    for rank, result in enumerate(sparse_results):
        cid = normalizeChunkID(result["chunk_id"])
        scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)

    return sorted(scores.keys(), key=lambda cid: scores[cid], reverse=True)

In [ ]:
# deduplicate articles so we don't stuff the prompt with same articles
def dedup_by_article(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen_articles = set()
    filtered = []

    for c in chunks:
        art_num = c.get("article_number")
        if art_num not in seen_articles:
            filtered.append(c)
            seen_articles.add(art_num)

    return filtered

In [ ]:
# Historical example: load a newly configured key from the environment.
# The previously exposed credential must be rotated by its owner.
gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


In [ ]:
# rewrite queries

REWRITE_SYSTEM = """Tu reformules une question pour améliorer la recherche documentaire
dans le Code de commerce marocain (loi n° 15-95).

Objectif :
Produire une reformulation utile pour la recherche, pas une paraphrase élégante.

Contraintes obligatoires :
- conserver strictement les entités juridiques clés de la question
- ne pas remplacer un terme juridique par un terme plus général ou voisin
- conserver les contraintes numériques et logiques, comme : "deux cas", "conditions", "délais", "qui", "quand"
- garder si possible les mots exacts du texte légal quand ils sont déjà présents dans la question
- éviter toute précision non demandée
- éviter les formules générales inutiles comme "conformément aux dispositions de la loi n° 15-95"
- ne produire qu'une seule requête
- sortie brute uniquement, sans explication ni ponctuation ajoutée inutile

Si la question contient déjà de bons termes juridiques, la reformulation doit rester très proche de l'original.
"""

def rewrite_query(query: str) -> str:
    prompt = f"""{REWRITE_SYSTEM}

Question originale :
{query}

Reformulation juridique :
"""

    response = gemini_client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
    )

    rewritten = (response.text or "").strip()
    return rewritten if rewritten else query

In [ ]:
# build prompts
SYSTEM = """Tu es un expert juridique spécialisé dans le Code de Commerce marocain (Loi n° 15-95).

Règles :
- Réponds uniquement à partir du contexte fourni.
- Réponds uniquement à la question posée, sans ajouter d'informations connexes non nécessaires.
- Cite uniquement les articles strictement nécessaires pour répondre.
- Si plusieurs articles sont fournis mais qu'un seul répond directement à la question, concentre-toi sur cet article.
- N'ajoute pas de remarque complémentaire sauf si elle est indispensable à la réponse.
- Si la réponse n'apparaît pas clairement dans le contexte, dis-le explicitement.
- N'invente ni article, ni règle.
- Réponds en français.
"""

def build_prompt(query: str, chunks: List[Dict[str, Any]]) -> str:
    context = "\n\n".join(
        f"[{c.get('context_header', '')}]\n"
        f"Article {c.get('article_number', 'N/A')}\n"
        f"{c.get('text', '')}"
        for c in chunks
    )

    return f"""{SYSTEM}

Contexte juridique :
{context}

Question juridique :
{query}

Réponse juridique :
"""

In [ ]:
# answer generation

def answer_with_gemini(query: str, chunks: List[Dict[str, Any]]) -> str:
    prompt = build_prompt(query, chunks)

    response = gemini_client.models.generate_content(
        model="gemini-3-flash-preview",
        contents=prompt,
    )

    return (response.text or "").strip()

In [ ]:
# inspect retrivals
def inspect_retrieval(query: str, use_rewrite: bool = True) -> Dict[str, Any]:
    rewritten_query = rewrite_query(query) if use_rewrite else query

    dense_results = denseSearch(rewritten_query, top_k=DENSE_TOP_K)
    sparse_results = bm25Search(rewritten_query, top_k=SPARSE_TOP_K)

    ranked_ids = reciprocal_rank_fusion(dense_results, sparse_results)

    retrieved_chunks = []
    for cid in ranked_ids:
        cid = normalizeChunkID(cid)
        if cid in chunk_lookup:
            retrieved_chunks.append(chunk_lookup[cid])
        else:
            dense_payload = next(
                (r["payload"] for r in dense_results if normalizeChunkID(r["chunk_id"]) == cid),
                None
            )
            if dense_payload:
                retrieved_chunks.append(dense_payload)
                
    retrieved_chunks = cross_encoder_rerank(rewritten_query, retrieved_chunks)

    if USE_ARTICLE_DEDUP:
        retrieved_chunks = dedup_by_article(retrieved_chunks)

    top_chunks = retrieved_chunks[:FINAL_TOP_K]

    return {
        "original_query": query,
        "rewritten_query": rewritten_query,
        "dense_results": dense_results,
        "sparse_results": sparse_results,
        "ranked_ids": ranked_ids,
        "top_chunks": top_chunks,
        "reranked_chunks": retrieved_chunks,

    }

In [ ]:
# full rag query
def rag_query(query: str, top_k: int = FINAL_TOP_K, use_rewrite: bool = True) -> Dict[str, Any]:
    debug = inspect_retrieval(query, use_rewrite=use_rewrite)
    top_chunks = debug["top_chunks"][:top_k]
    final_answer = answer_with_gemini(query, top_chunks)

    return {
        "query": query,
        "rewritten_query": debug["rewritten_query"],
        "top_chunks": top_chunks,
        "answer": final_answer,
        "dense_results": debug["dense_results"],
        "sparse_results": debug["sparse_results"],
        "ranked_ids": debug["ranked_ids"],
    }

In [ ]:
# protify print helpers

def print_dense_results(results: List[Dict[str, Any]], n: int = 5) -> None:
    print("\nDENSE RESULTS")
    for r in results[:n]:
        payload = r.get("payload", {})
        print("-" * 80)
        print("chunk_id:", r["chunk_id"])
        print("score:", round(r["score"], 4))
        print("article:", payload.get("article_number"))
        print("context:", payload.get("context_header", "")[:120])

def print_sparse_results(results: List[Dict[str, Any]], n: int = 5) -> None:
    print("\nBM25 RESULTS")
    for r in results[:n]:
        doc = chunk_lookup.get(normalizeChunkID(r["chunk_id"]), {})
        print("-" * 80)
        print("chunk_id:", r["chunk_id"])
        print("score:", round(r["score"], 4))
        print("article:", doc.get("article_number"))
        print("context:", doc.get("context_header", "")[:120])

def print_top_chunks(chunks: List[Dict[str, Any]]) -> None:
    print("\nTOP CHUNKS USED FOR GENERATION")
    for i, c in enumerate(chunks, 1):
        print("=" * 100)
        print(f"[{i}] Article {c.get('article_number')}")
        print(c.get("context_header", ""))
        print("-" * 100)
        print(c.get("text", "")[:900])

In [ ]:
# Test query rewriting only
test_query = "Quand une obligation commerciale devient-elle prescrite ?"

rewritten = rewrite_query(test_query)
print("Original :", test_query)
print("Rewritten:", rewritten)

In [ ]:
#Test retrieval only
test_query = "Dans quels deux cas un commerçant ou une personne morale est-il radié d’office ?"

debug = inspect_retrieval(test_query, use_rewrite=True)

print("Original query :", debug["original_query"])
print("Rewritten query:", debug["rewritten_query"])

print_dense_results(debug["dense_results"], n=5)
print_sparse_results(debug["sparse_results"], n=5)
print_top_chunks(debug["top_chunks"])

In [ ]:
# Test full generation
test_query = "Dans quels deux cas un commerçant ou une personne morale est-il radié d’office ?"

result = rag_query(test_query, top_k=5, use_rewrite=True)

print("QUESTION")
print(result["query"])

print("\nREWRITTEN QUERY")
print(result["rewritten_query"])

print_top_chunks(result["top_chunks"])

print("\nFINAL ANSWER")
print(result["answer"])